In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp02
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp02/ex_2.py ──
"""
Shared infrastructure for MLFP02 Exercise 2 — Parameter Estimation and Inference.

Contains: Singapore economic data loading, log-likelihood objectives for
Normal / Student-t / Laplace, profile LR CI helpers, bootstrap utilities,
plot output directory and save helper.

Technique-specific narrative (which scenario, which interpretation) belongs
in the per-technique files in ``modules/mlfp02/solutions/ex_2/``.
"""

from pathlib import Path
from typing import Callable

import numpy as np
import plotly.graph_objects as go
import polars as pl
from scipy import stats
from scipy.optimize import minimize


# ════════════════════════════════════════════════════════════════════════
# OUTPUT DIRECTORY
# ════════════════════════════════════════════════════════════════════════

OUTPUT_DIR = Path("outputs") / "mlfp02_ex2_mle_theory"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def save_figure(fig: go.Figure, filename: str) -> Path:
    """Write a Plotly figure to the exercise output directory."""
    path = OUTPUT_DIR / filename
    fig.write_html(str(path))
    return path


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Singapore economic indicators
# ════════════════════════════════════════════════════════════════════════

SINGAPORE_ECON_DATASET = ("mlfp01", "economic_indicators.csv")


def load_singapore_econ() -> pl.DataFrame:
    """Load Singapore economic indicators (GDP, inflation, unemployment)."""
    loader = MLFPDataLoader()
    return loader.load(*SINGAPORE_ECON_DATASET)


def extract_series(df: pl.DataFrame, column: str) -> np.ndarray:
    """Drop nulls and return a float64 numpy array for a given column."""
    return df[column].drop_nulls().to_numpy().astype(np.float64)


def load_gdp_growth() -> np.ndarray:
    """Convenience: GDP growth (annualised quarterly %) as a 1-D numpy array."""
    return extract_series(load_singapore_econ(), "gdp_growth_pct")


# ════════════════════════════════════════════════════════════════════════
# LIKELIHOOD OBJECTIVES
# ════════════════════════════════════════════════════════════════════════
#
# All objectives are NEGATIVE log-likelihoods so they can be passed
# directly to scipy.optimize.minimize. Location/scale parameters are
# reparameterised where needed (log-sigma) to keep the optimiser in a
# valid region without explicit bounds.


def neg_log_likelihood_normal(params: np.ndarray, x: np.ndarray) -> float:
    """-log L for X ~ N(mu, sigma^2). params = [mu, log_sigma]."""
    mu, log_sigma = params
    sigma = np.exp(log_sigma)
    if sigma <= 0:
        return np.inf
    return float(-np.sum(stats.norm.logpdf(x, loc=mu, scale=sigma)))


def neg_log_likelihood_t(params: np.ndarray, x: np.ndarray) -> float:
    """-log L for X ~ t(df, mu, scale). params = [df, mu, scale]."""
    df, mu, scale = params
    if df <= 0 or scale <= 0:
        return np.inf
    return float(-np.sum(stats.t.logpdf(x, df=df, loc=mu, scale=scale)))


def fit_normal_mle(x: np.ndarray) -> dict:
    """Fit Normal MLE numerically via L-BFGS-B. Returns mu, sigma, loglik, result."""
    x0 = np.array([float(x.mean()), float(np.log(x.std() + 1e-8))])
    result = minimize(
        neg_log_likelihood_normal,
        x0,
        args=(x,),
        method="L-BFGS-B",
        options={"maxiter": 1000, "ftol": 1e-12},
    )
    return {
        "mu": float(result.x[0]),
        "sigma": float(np.exp(result.x[1])),
        "loglik": float(-result.fun),
        "converged": bool(result.success),
        "result": result,
    }


def fit_student_t_mle(x: np.ndarray) -> dict:
    """Fit Student-t MLE via Nelder-Mead. Returns df, mu, scale, loglik."""
    x0 = np.array([5.0, float(x.mean()), float(x.std() + 1e-8)])
    result = minimize(neg_log_likelihood_t, x0, args=(x,), method="Nelder-Mead")
    return {
        "df": float(result.x[0]),
        "mu": float(result.x[1]),
        "scale": float(result.x[2]),
        "loglik": float(-result.fun),
        "converged": bool(result.success),
        "result": result,
    }


# ════════════════════════════════════════════════════════════════════════
# FISHER INFORMATION + CONFIDENCE INTERVALS
# ════════════════════════════════════════════════════════════════════════
#
# For N(mu, sigma^2):
#   Var(mu_hat)   = sigma^2 / n         =>  SE(mu_hat)  = sigma / sqrt(n)
#   Var(sigma_hat) ~ sigma^2 / (2n)     =>  SE(sigma_hat) = sigma / sqrt(2n)
#
# These come directly from inverting the Fisher information matrix at
# the MLE.


def normal_fisher_standard_errors(sigma: float, n: int) -> tuple[float, float]:
    """Return (SE_mu, SE_sigma) from Fisher information at the Normal MLE."""
    se_mu = sigma / np.sqrt(n)
    se_sigma = sigma / np.sqrt(2 * n)
    return float(se_mu), float(se_sigma)


def wald_ci(estimate: float, se: float, alpha: float = 0.05) -> tuple[float, float]:
    """Symmetric Wald CI: estimate ± z_{1-alpha/2} * SE."""
    z = float(stats.norm.ppf(1 - alpha / 2))
    return (estimate - z * se, estimate + z * se)


def profile_lr_ci_normal_mu(
    x: np.ndarray,
    mu_hat: float,
    sigma_hat: float,
    loglik_at_mle: float,
    alpha: float = 0.05,
    grid_width_in_se: float = 4.0,
    n_grid: int = 500,
) -> tuple[tuple[float, float], np.ndarray, np.ndarray]:
    """Profile likelihood 1-alpha CI for the Normal mean.

    The CI is the set of mu where 2*(loglik_at_mle - loglik(mu)) < chi^2_{1-alpha, df=1}.

    Returns (ci, mu_grid, loglik_values) so the caller can plot the profile.
    """
    n = len(x)
    se_mu = sigma_hat / np.sqrt(n)
    threshold = float(stats.chi2.ppf(1 - alpha, df=1)) / 2.0

    mu_grid = np.linspace(
        mu_hat - grid_width_in_se * se_mu,
        mu_hat + grid_width_in_se * se_mu,
        n_grid,
    )
    loglik_values = np.array(
        [-neg_log_likelihood_normal([mu, np.log(sigma_hat)], x) for mu in mu_grid]
    )
    lr_values = loglik_at_mle - loglik_values
    mask = lr_values <= threshold
    if mask.any():
        ci = (float(mu_grid[mask][0]), float(mu_grid[mask][-1]))
    else:
        # Fallback: Wald CI
        ci = wald_ci(mu_hat, se_mu, alpha)
    return ci, mu_grid, loglik_values


# ════════════════════════════════════════════════════════════════════════
# MAP — NORMAL LIKELIHOOD + NORMAL PRIOR ON THE MEAN
# ════════════════════════════════════════════════════════════════════════


def make_map_objective(
    prior_mean: float, prior_std: float
) -> Callable[[np.ndarray, np.ndarray], float]:
    """Return a neg_map_objective(params, x) closure for the given Normal prior on mu."""

    def neg_map_objective(params: np.ndarray, x: np.ndarray) -> float:
        mu, log_sigma = params
        sigma = np.exp(log_sigma)
        if sigma <= 0:
            return np.inf
        nll = -np.sum(stats.norm.logpdf(x, loc=mu, scale=sigma))
        neg_log_prior = -stats.norm.logpdf(mu, loc=prior_mean, scale=prior_std)
        return float(nll + neg_log_prior)

    return neg_map_objective


def fit_normal_map(x: np.ndarray, prior_mean: float, prior_std: float) -> dict:
    """Fit MAP for Normal likelihood with a Normal(prior_mean, prior_std^2) prior on mu."""
    neg_map = make_map_objective(prior_mean, prior_std)
    x0 = np.array([float(x.mean()), float(np.log(x.std() + 1e-8))])
    result = minimize(
        neg_map,
        x0,
        args=(x,),
        method="L-BFGS-B",
        options={"maxiter": 1000, "ftol": 1e-12},
    )
    return {
        "mu": float(result.x[0]),
        "sigma": float(np.exp(result.x[1])),
        "converged": bool(result.success),
        "result": result,
    }


# ════════════════════════════════════════════════════════════════════════
# BOOTSTRAP UTILITIES
# ════════════════════════════════════════════════════════════════════════


def bootstrap_statistic(
    x: np.ndarray,
    statistic: Callable[[np.ndarray], float],
    n_boot: int = 10_000,
    seed: int | None = 42,
) -> np.ndarray:
    """Nonparametric bootstrap: resample x with replacement, apply statistic."""
    rng = np.random.default_rng(seed)
    n = len(x)
    out = np.empty(n_boot, dtype=np.float64)
    for i in range(n_boot):
        out[i] = statistic(rng.choice(x, size=n, replace=True))
    return out


def bootstrap_percentile_ci(
    boot_samples: np.ndarray, alpha: float = 0.05
) -> tuple[float, float]:
    lo = float(np.percentile(boot_samples, 100 * alpha / 2))
    hi = float(np.percentile(boot_samples, 100 * (1 - alpha / 2)))
    return lo, hi


# ════════════════════════════════════════════════════════════════════════
# AIC / BIC
# ════════════════════════════════════════════════════════════════════════


def aic(k: int, loglik: float) -> float:
    return 2 * k - 2 * loglik


def bic(k: int, loglik: float, n: int) -> float:
    return k * float(np.log(n)) - 2 * loglik


# ════════════════════════════════════════════════════════════════════════
# DEFAULTS — SAMPLE SIZES, SEEDS, PRIOR VALUES
# ════════════════════════════════════════════════════════════════════════
#
# These constants are referenced by multiple technique files. Change them
# once here and every file picks up the update.

DEFAULT_SEED: int = 42
DEFAULT_N_BOOT: int = 10_000
DEFAULT_N_CLT_REPS: int = 5000

# Singapore prior on quarterly GDP growth — long-run open-economy estimate
# that we use to illustrate MAP shrinkage.
GDP_PRIOR_MEAN: float = 3.5  # %
GDP_PRIOR_STD: float = 1.5  # %




# ════════════════════════════════════════════════════════════════════════
# MLFP02 — Exercise 2.1: Central Limit Theorem and Sampling
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Profile distribution shape before fitting parametric models
#     (Shapiro-Wilk, skewness, excess kurtosis)
#   - Distinguish population parameters from sample statistics and
#     explain why Bessel's correction (ddof=1) fixes variance bias
#   - Simulate the Central Limit Theorem: observe how the sampling
#     distribution of the mean becomes Normal regardless of population
#   - Visualise how CLT convergence depends on sample size and shape
#
# PREREQUISITES: MLFP02 Exercise 1 (probability, Bayesian thinking)
# ESTIMATED TIME: ~35 minutes
#
# TASKS (5-phase R10):
#   1. Theory — population vs sample, CLT statement
#   2. Build — three synthetic populations (Exponential, Uniform, Bimodal)
#   3. Train — run CLT simulations at different sample sizes
#   4. Visualise — CLT sampling distributions and Bessel's correction
#   5. Apply — MAS quarterly GDP volatility reporting
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats



## THEORY — Population vs Sample, and the CLT


In [ ]:
#
# POPULATION PARAMETERS vs SAMPLE STATISTICS:
#   Population: mu (true mean), sigma (true std) — fixed but unknown.
#   Sample:     x-bar (sample mean), s (sample std) — random.
#
# KEY INSIGHT: MLE variance (ddof=0) underestimates the population
# variance because the sample mean "uses up" one degree of freedom.
# Bessel's correction divides by (n-1) to give an unbiased estimator.
#
# CENTRAL LIMIT THEOREM:
#   Regardless of the population distribution, the sampling distribution
#   of x-bar approaches N(mu, sigma^2/n) as n -> infinity.



## TASK 1 — Data Profiling: Distribution Shape Assessment


In [ ]:
econ = load_singapore_econ()
gdp_growth = extract_series(econ, "gdp_growth_pct")
inflation = extract_series(econ, "inflation_rate")
unemployment = extract_series(econ, "unemployment_rate")

n_gdp = len(gdp_growth)

print("=" * 70)
print("  MLFP02 Exercise 2.1: CLT and Sampling")
print("=" * 70)
print(f"\n  GDP growth: {n_gdp} observations")
print(f"  Range: {gdp_growth.min():.2f}% to {gdp_growth.max():.2f}%")
print(f"  Sample mean: {gdp_growth.mean():.3f}%")
print(f"  Sample std:  {gdp_growth.std():.3f}%")

# TODO: Run the Shapiro-Wilk normality test on gdp_growth.
# Hint: stats.shapiro(gdp_growth) returns (statistic, p_value)
shapiro_stat, shapiro_p = ____

print(f"\n=== Distribution Shape Assessment ===")
print(f"Shapiro-Wilk test: W={shapiro_stat:.4f}, p={shapiro_p:.4f}")
if shapiro_p > 0.05:
    print("Cannot reject normality — Normal likelihood is plausible")
else:
    print("Normality rejected — consider heavier-tailed distributions (t, skew-Normal)")

# TODO: Compute the skewness and excess kurtosis of gdp_growth.
# Hint: stats.skew(data) and stats.kurtosis(data)
skew = ____
kurt = ____
print(f"Skewness: {skew:.3f} (|skew|>1 suggests non-Normal)")
print(f"Excess kurtosis: {kurt:.3f} (>0 -> heavier tails than Normal)")

# Descriptive statistics for all three series
for name, arr in [
    ("GDP growth", gdp_growth),
    ("Inflation", inflation),
    ("Unemployment", unemployment),
]:
    print(
        f"\n{name}: mean={arr.mean():.3f}, std={arr.std():.3f}, "
        f"min={arr.min():.3f}, max={arr.max():.3f}, n={len(arr)}"
    )



### Checkpoint 1


In [ ]:
assert 0 <= shapiro_p <= 1, "Shapiro-Wilk p-value must be between 0 and 1"
assert n_gdp > 10, f"Expected substantial GDP history, got {n_gdp} observations"
print("\n--- Checkpoint 1 passed --- data profiled and normality assessed\n")



## TASK 2 — BUILD: Bessel's Correction Simulation


In [ ]:
print(f"\n=== Population vs Sample Variance ===")

rng = np.random.default_rng(seed=DEFAULT_SEED)
true_mu = 3.0
true_sigma = 5.0
n_sim_repeats = DEFAULT_N_CLT_REPS
sample_size_demo = 10

mle_vars = []
unbiased_vars = []

for _ in range(n_sim_repeats):
    sample = rng.normal(true_mu, true_sigma, size=sample_size_demo)
    # TODO: Compute the MLE variance (ddof=0) and unbiased variance (ddof=1)
    # Hint: sample.var(ddof=0) and sample.var(ddof=1)
    mle_vars.append(____)
    unbiased_vars.append(____)

mle_var_mean = np.mean(mle_vars)
unbiased_var_mean = np.mean(unbiased_vars)
true_var = true_sigma**2

print(f"True sigma^2 = {true_var:.2f}")
print(f"n per sample: {sample_size_demo}")
print(f"Simulations: {n_sim_repeats:,}")
print(
    f"E[MLE var] (ddof=0):     {mle_var_mean:.2f} (bias = {mle_var_mean - true_var:+.2f})"
)
print(
    f"E[unbiased s^2] (ddof=1): {unbiased_var_mean:.2f} (bias = {unbiased_var_mean - true_var:+.2f})"
)
print(
    f"Ratio MLE/true: {mle_var_mean / true_var:.4f} (expected: {(sample_size_demo-1)/sample_size_demo:.4f})"
)



### Checkpoint 2


In [ ]:
assert mle_var_mean < true_var, "MLE variance should underestimate on average"
assert (
    abs(unbiased_var_mean - true_var) < true_var * 0.05
), "Unbiased variance should be within 5% of true variance"
print("\n--- Checkpoint 2 passed --- Bessel's correction demonstrated\n")



## TASK 3 — TRAIN: Central Limit Theorem Simulation


In [ ]:
print(f"\n=== Central Limit Theorem Simulation ===")

n_clt_samples = DEFAULT_N_CLT_REPS
sample_sizes_clt = [5, 15, 30, 100]

# TODO: Create three synthetic populations:
# 1. Exponential(scale=1.0) with 100,000 samples
# 2. Uniform(0, 10) with 100,000 samples
# 3. Bimodal: concatenate N(2, 0.5) with 50K and N(8, 0.5) with 50K
# Hint: rng.exponential(), rng.uniform(), rng.normal(), np.concatenate()
population_exp = ____
population_unif = ____
pop_bimodal = np.concatenate(
    [
        ____,
        ____,
    ]
)

populations = {
    "Exponential(1)": population_exp,
    "Uniform(0,10)": population_unif,
    "Bimodal N(2,0.5)+N(8,0.5)": pop_bimodal,
}

for pop_name, pop in populations.items():
    print(f"\n--- {pop_name} ---")
    print(
        f"Population: mean={pop.mean():.3f}, std={pop.std():.3f}, skew={stats.skew(pop):.3f}"
    )

    for n_s in sample_sizes_clt:
        # TODO: Draw n_clt_samples sets of size n_s (with replacement),
        # compute the mean of each set, store in sample_means.
        # Hint: rng.choice(pop, size=n_s, replace=True).mean()
        sample_means = np.array([____ for _ in range(n_clt_samples)])
        sm_mean = sample_means.mean()
        sm_std = sample_means.std()
        sm_skew = stats.skew(sample_means)
        theoretical_se = pop.std() / np.sqrt(n_s)

        sw_stat, sw_p = stats.shapiro(
            rng.choice(sample_means, size=min(500, n_clt_samples), replace=False)
        )

        print(
            f"  n={n_s:>3}: mean(x-bar)={sm_mean:.3f}, "
            f"SE(x-bar)={sm_std:.3f} (theory: {theoretical_se:.3f}), "
            f"skew={sm_skew:.3f}, Shapiro p={sw_p:.3f}"
        )



### Checkpoint 3


In [ ]:
exp_means_100 = np.array(
    [rng.choice(population_exp, size=100, replace=True).mean() for _ in range(2000)]
)
_, sw_p_100 = stats.shapiro(rng.choice(exp_means_100, size=200, replace=False))
assert (
    sw_p_100 > 0.01
), f"CLT: sampling distribution of mean should be ~Normal for n=100, Shapiro p={sw_p_100}"
print("\n--- Checkpoint 3 passed --- CLT demonstrated across three population shapes\n")



## TASK 4 — VISUALISE: CLT Convergence


In [ ]:
fig1 = make_subplots(rows=1, cols=3, subplot_titles=["n=5", "n=30", "n=100"])
for i, ns in enumerate([5, 30, 100]):
    sample_means = [rng.choice(population_exp, size=ns).mean() for _ in range(3000)]
    fig1.add_trace(
        go.Histogram(
            x=sample_means, nbinsx=40, name=f"n={ns}", opacity=0.7, showlegend=True
        ),
        row=1,
        col=i + 1,
    )
fig1.update_layout(
    title="CLT: Sampling Distribution of Mean from Exponential(1)", height=350
)
save_figure(fig1, "ex2_01_clt_demonstration.html")
print("Saved: ex2_01_clt_demonstration.html")



### Checkpoint 4


In [ ]:
print("\n--- Checkpoint 4 passed --- CLT visualisation saved\n")



## TASK 5 — APPLY: MAS Quarterly GDP Volatility Reporting


In [ ]:
# MAS publishes quarterly GDP statistics. The standard error determines
# how precisely we know the average growth rate. Bessel correction
# matters with only ~40-60 quarterly observations.

print(f"\n=== APPLY: MAS GDP Volatility Reporting ===")

# TODO: Compute the MLE sigma (ddof=0) and unbiased sigma (ddof=1)
# for the real GDP growth data. Then compute SE = sigma / sqrt(n).
# Hint: gdp_growth.std(ddof=0) and gdp_growth.std(ddof=1)
sigma_mle = ____
sigma_unbiased = ____
se_mle = sigma_mle / np.sqrt(n_gdp)
se_unbiased = sigma_unbiased / np.sqrt(n_gdp)

print(f"GDP observations: {n_gdp} quarters")
print(f"MLE sigma: {sigma_mle:.3f}%   ->  SE(mean) = {se_mle:.3f}%")
print(f"Unbiased sigma: {sigma_unbiased:.3f}%  ->  SE(mean) = {se_unbiased:.3f}%")
print(
    f"Difference: {(sigma_unbiased - sigma_mle):.4f}% "
    f"({(sigma_unbiased - sigma_mle) / sigma_mle * 100:.2f}% wider)"
)
print(
    f"\nFor MAS fan chart: 95% CI for mean growth = "
    f"[{gdp_growth.mean() - 1.96 * se_unbiased:.2f}%, "
    f"{gdp_growth.mean() + 1.96 * se_unbiased:.2f}%]"
)
print("Using the biased MLE would produce a CI that is too narrow,")
print("giving policymakers false confidence in the growth forecast.")



### Checkpoint 5


In [ ]:
assert sigma_unbiased > sigma_mle, "Unbiased sigma must exceed MLE sigma"
print("\n--- Checkpoint 5 passed --- MAS application complete\n")



## REFLECTION


- Population vs sample: parameter (mu, sigma) vs statistic (x-bar, s)
  - Bessel's correction: MLE variance is biased low by factor (n-1)/n
  - Central Limit Theorem: x-bar -> Normal regardless of population
    shape, demonstrated for Exponential, Uniform, and Bimodal
  - Shapiro-Wilk test to assess normality before fitting models
  - Real-world impact: MAS GDP reporting with correct volatility bands

  NEXT: In 02_mle_fisher.py, you'll write and optimise a log-likelihood
  function using scipy.optimize, then compute standard errors from the
  Fisher information matrix and compare Wald vs profile likelihood CIs.


In [ ]:
print("=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

print("--- Exercise 2.1 complete --- CLT and Sampling")

